In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[-0.0071,  0.0239, -0.1153,  0.2007,  0.0107,  0.1669,  0.1043, -0.2106,
         -0.0280, -0.1782],
        [ 0.0135, -0.0241, -0.1183,  0.1304,  0.1017,  0.1590,  0.1856, -0.3570,
          0.0160, -0.2322]], grad_fn=<AddmmBackward0>)

多层感知器中，通过实例化nn.Sequential来构建我们的模型，nn.Sequential定义了一种特殊的Module，其中两个全连接层都是Linear类的实例，Linear类本身就是Module的子类。

通过net(X)调用我们的模型来获得模型的输出。

sequential里已经写好了forward（），net（X）时会自动执行

In [ ]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

net = MLP()
net(X)

可以自定义nn.Module来搭建模型，其中自定义前向传播函数，它以X作为输入，计算带有激活函数的隐藏表示，并输出其未规范化的输出值。

其中两个层都是实例变量，所以可以实例化多个模型，每次调用前向传播函数时会调用这些层。

module的优点是多功能性，可以创建复杂模型

顺序块

构建Mysequential，仅需定义两个函数：1.将块逐个加进列表 2.前向传播函数

In [2]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

当MySequential的前向传播函数被调用时， 每个添加的块都按照它们被添加的顺序执行。 现在可以使用我们的MySequential类重新实现多层感知机。

In [3]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.0923,  0.0896,  0.1313, -0.0056,  0.0078,  0.1629,  0.0167, -0.1965,
         -0.1252, -0.0224],
        [ 0.1164,  0.0455,  0.0649, -0.0354, -0.0151,  0.1290,  0.0100, -0.1835,
         -0.1939,  0.0692]], grad_fn=<AddmmBackward0>)

在前向传播函数中执行代码

不是所有的架构都是简单的顺序架构。 当需要更强的灵活性时，我们需要定义自己的块。

如果想要合并一个不是上层结果，也不是更新参数的参数，即常熟参数，例如，我们需要一个计算函数
$f(\mathbf{x},\mathbf{w}) = c \cdot \mathbf{w}^\top \mathbf{x}$的层，
其中$\mathbf{x}$是输入，
$\mathbf{w}$是参数，
$c$是某个在优化过程中没有更新的指定常量。

In [ ]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

self.rand_weight是固定隐藏层，随机初始化不会更新，然后将其输出通过一个全连接层self.linear

混合搭配各种组合块的方法

In [ ]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

## 练习

1. 如果将`MySequential`中存储块的方式更改为Python列表，会出现什么样的问题？

原来的MySequential通过self._modules[str(idx)] = module将module注册到模型里，若改成普通列表就是self.layers = []，只是将曾放进列表，没有告诉pytorch这些是层，所以参数不会被正确跟踪，要先把模块按照类似于self.fc1 = nn.Linear(20, 64)的方式注册为层，将其交给pytorch，从而其中参数可以被识别

1. 实现一个块，它以两个块为参数，例如`net1`和`net2`，并返回前向传播中两个网络的串联输出。这也被称为平行块。

In [4]:
import torch
from torch import nn

class ParallelBlock(nn.Module):
    def __init__(self, net1, net2, dim=1):
        super().__init__()
        self.net1 = net1
        self.net2 = net2
        self.dim = dim  # 按哪个维度拼接

    def forward(self, X):
        y1 = self.net1(X)
        y2 = self.net2(X)
        return torch.cat((y1, y2), dim=self.dim)

In [5]:
net1 = nn.Sequential(
    nn.Linear(20, 10),
    nn.ReLU()
)

net2 = nn.Sequential(
    nn.Linear(20, 6),
    nn.ReLU()
)

parallel_net = ParallelBlock(net1, net2, dim=1)

X = torch.randn(4, 20)
Y = parallel_net(X)

print(Y.shape)

torch.Size([4, 16])


net1(X) 输出形状是 (4, 10)
net2(X) 输出形状是 (4, 6)
拼接后就是 (4, 16)

1. 假设我们想要连接同一网络的多个实例。实现一个函数，该函数生成同一个块的多个实例，并在此基础上构建更大的网络。

In [6]:
class SimpleBlock(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.relu = nn.ReLU()

    def forward(self, X):
        return self.relu(self.linear(X))

先创建一个普通的块

In [7]:
def repeated_block(block_cls, num_blocks, *args, **kwargs):
    blocks = []
    for _ in range(num_blocks):
        blocks.append(block_cls(*args, **kwargs))
    return nn.Sequential(*blocks)

写一个函数，能够生成多个块，并将其收集起来用nn.Sequential串成网络

In [8]:
net = repeated_block(SimpleBlock, 3, 20, 20)

X = torch.randn(4, 20)
Y = net(X)
print(Y.shape)

torch.Size([4, 20])


In [9]:
big_net = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    repeated_block(SimpleBlock, 3, 20, 20),
    nn.Linear(20, 1)
)

X = torch.randn(5, 10)
Y = big_net(X)
print(Y.shape)

torch.Size([5, 1])


再将上面的多层块进行复用，构建更大的网络